# Base 4 — temperatura em Jena

Pipeline SARIMAX adaptada de `docs/Pipes de exemplo/atividade_sarimax.ipynb`. Resultados ficam incorporados neste notebook; nenhum CSV/JSON auxiliar é criado.

In [ ]:
from pathlib import Path
import itertools, platform, time, warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import stats
from sklearn.metrics import mean_absolute_error, mean_squared_error, median_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller, kpss

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 100)
warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
BASE_NUMBER=4
TIME_COL='Date Time'
TARGET_COL='T (degC)'
FREQUENCY='10min'
TRAIN_RATIO=0.8
SEASONAL_PERIOD=144
ACF_LAGS=288
LJUNG_LAGS=[1, 6, 144, 288]
RANDOM_STATE=42
SMOKE_TEST=True
np.random.seed(RANDOM_STATE)

In [ ]:
def encontrar_grupo(numero):
    atual = Path.cwd().resolve()
    for raiz in [atual, *atual.parents]:
        pasta = raiz / "trabalho" / "bases" / f"grupo{numero}"
        if pasta.is_dir(): return pasta
        pasta = raiz / f"grupo{numero}"
        if pasta.is_dir(): return pasta
    raise FileNotFoundError(f"grupo{numero} não encontrado")

def carregar_base(pasta):
    treino = pasta / f"base{BASE_NUMBER}_treino_preparada.csv"
    teste = pasta / f"base{BASE_NUMBER}_teste_preparada.csv"
    if treino.exists() and teste.exists():
        return pd.concat([pd.read_csv(treino), pd.read_csv(teste)], ignore_index=True)
    for nome in [f"grupo{BASE_NUMBER}.csv", "grupo5_new.csv"]:
        arquivo = pasta / nome
        if arquivo.exists(): return pd.read_csv(arquivo)
    raise FileNotFoundError("CSV da base não encontrado")

def mape_seguro(y, p):
    y, p = np.asarray(y,float), np.asarray(p,float); m = y != 0
    return float(np.mean(np.abs((y[m]-p[m])/y[m]))*100) if m.any() else np.nan

def metricas(y, p):
    y, p = np.asarray(y,float), np.asarray(p,float)
    return {"MAE": mean_absolute_error(y,p), "RMSE": mean_squared_error(y,p)**.5,
            "MedAE": median_absolute_error(y,p), "R2": r2_score(y,p), "MAPE_pct": mape_seguro(y,p)}

def escalar_exog(x_treino, x_futuro):
    if x_treino is None: return None, None, None
    scaler = StandardScaler()
    a = pd.DataFrame(scaler.fit_transform(x_treino), index=x_treino.index, columns=x_treino.columns)
    b = pd.DataFrame(scaler.transform(x_futuro), index=x_futuro.index, columns=x_futuro.columns)
    return a, b, scaler

def ajustar(y, x, order, seasonal_order, trend="n", maxiter=100):
    inicio=time.perf_counter()
    with warnings.catch_warnings(record=True) as capturados:
        warnings.simplefilter("always")
        resultado=SARIMAX(y, exog=x, order=order, seasonal_order=seasonal_order, trend=trend,
                          enforce_stationarity=False, enforce_invertibility=False).fit(disp=False,maxiter=maxiter)
    return resultado, bool(resultado.mle_retvals.get("converged",True)), time.perf_counter()-inicio, " | ".join(str(w.message) for w in capturados[:3])

def ajustar_prever(treino, futuro, order, seasonal_order, exog_cols=(), trend="n"):
    xt, xf, scaler = escalar_exog(treino[list(exog_cols)] if exog_cols else None,
                                  futuro[list(exog_cols)] if exog_cols else None)
    res, convergiu, segundos, avisos = ajustar(treino[TARGET_COL], xt, order, seasonal_order, trend)
    pred = res.get_forecast(steps=len(futuro), exog=xf).predicted_mean
    pred.index=futuro.index
    return {"resultado":res,"previsao":pred,"convergiu":convergiu,"segundos":segundos,"avisos":avisos,"scaler":scaler}

## 1. SARIMAX — ingestão, corte temporal e preparação das exógenas

In [ ]:
BASE_DIR=encontrar_grupo(BASE_NUMBER)
df=carregar_base(BASE_DIR)
df[TIME_COL]=pd.to_datetime(df[TIME_COL],errors="coerce",dayfirst=(BASE_NUMBER==4))
df[TARGET_COL]=pd.to_numeric(df[TARGET_COL],errors="coerce")
df=df.dropna(subset=[TIME_COL]).sort_values(TIME_COL)
numericas=df.select_dtypes(include=np.number).columns.tolist()
df=df.groupby(TIME_COL,sort=True)[numericas].mean()
grade=pd.date_range(df.index.min(),df.index.max(),freq=FREQUENCY)
df=df.reindex(grade)
df["hour_sin"]=np.sin(2*np.pi*df.index.hour/24); df["hour_cos"]=np.cos(2*np.pi*df.index.hour/24)
df["dow_sin"]=np.sin(2*np.pi*df.index.dayofweek/7); df["dow_cos"]=np.cos(2*np.pi*df.index.dayofweek/7)
df["month_sin"]=np.sin(2*np.pi*(df.index.month-1)/12); df["month_cos"]=np.cos(2*np.pi*(df.index.month-1)/12)
EXOG_COLS=["hour_sin","hour_cos","dow_sin","dow_cos","month_sin","month_cos"]
corte=int(df[TARGET_COL].notna().sum()*TRAIN_RATIO)
inicio_teste=df[TARGET_COL].dropna().index[corte]
treino=df.loc[df.index<inicio_teste].copy(); teste=df.loc[df.index>=inicio_teste].copy()
teste_observado=teste.dropna(subset=[TARGET_COL]).copy()
print("Base:",BASE_DIR); print("Treino:",treino.index.min(),treino.index.max(),treino[TARGET_COL].notna().sum())
print("Teste:",teste_observado.index.min(),teste_observado.index.max(),len(teste_observado))
display(df[[TARGET_COL,*EXOG_COLS]].head())

In [ ]:
serie=treino[TARGET_COL].dropna()
print("ADF p-valor:",adfuller(serie,autolag="AIC")[1]); print("KPSS p-valor:",kpss(serie,regression="c",nlags="auto")[1])
lags=min(ACF_LAGS,max(1,len(serie)//3))
fig,ax=plt.subplots(1,2,figsize=(14,4)); plot_acf(serie,lags=lags,ax=ax[0]); plot_pacf(serie,lags=min(lags,50),method="ywm",ax=ax[1]); plt.show()

## 2. Otimização SARIMAX — ordens, sazonalidade, BIC e convergência

In [ ]:
validacao_n=min(max(2*SEASONAL_PERIOD,20),max(20,len(treino)//5))
ajuste=treino.iloc[:-validacao_n]; validacao=treino.iloc[-validacao_n:]
if SMOKE_TEST:
    ordens=[(1,0,1),(1,1,1),(2,1,1)]
    sazonais=[(0,0,0,0),(1,0,1,SEASONAL_PERIOD)]
else:
    ordens=list(itertools.product(range(3),range(2),range(3)))
    sazonais=[(0,0,0,0)]+[(P,D,Q,SEASONAL_PERIOD) for P,D,Q in itertools.product(range(2),range(2),range(2))]
conjuntos={"SARIMA_sem_exogenas":[],"SARIMAX_calendario":EXOG_COLS}
linhas=[]
for order,seasonal_order,(familia,cols) in itertools.product(ordens,sazonais,conjuntos.items()):
    try:
        out=ajustar_prever(ajuste,validacao,order,seasonal_order,cols)
        alvo=validacao[TARGET_COL].notna(); score=metricas(validacao.loc[alvo,TARGET_COL],out["previsao"].loc[alvo])
        linhas.append({"familia":familia,"order":order,"seasonal_order":seasonal_order,"exogenas":cols,
                       "BIC":out["resultado"].bic,"convergiu":out["convergiu"],"segundos":out["segundos"],"avisos":out["avisos"],**score})
    except Exception as erro:
        linhas.append({"familia":familia,"order":order,"seasonal_order":seasonal_order,"exogenas":cols,"erro":str(erro),"MAE":np.inf})
busca=pd.DataFrame(linhas).sort_values(["MAE","BIC"],na_position="last").reset_index(drop=True)
display(busca)
validos=busca[(busca["MAE"]<np.inf)&busca["convergiu"].fillna(False)]
melhor=(validos if len(validos) else busca[busca["MAE"]<np.inf]).iloc[0]
display(melhor.to_frame("melhor configuração"))

## 3. MAE e demais métricas SARIMAX

In [ ]:
final=ajustar_prever(treino,teste,tuple(melhor.order),tuple(melhor.seasonal_order),melhor.exogenas)
pred=final["previsao"].loc[teste_observado.index]
previsoes=pd.DataFrame({"y_true":teste_observado[TARGET_COL],"y_pred_sarimax":pred})
previsoes["residuo_sarimax"]=previsoes.y_true-previsoes.y_pred_sarimax
previsoes["erro_absoluto_sarimax"]=previsoes.residuo_sarimax.abs()
resumo={**metricas(previsoes.y_true,previsoes.y_pred_sarimax),"BIC":final["resultado"].bic,
        "convergiu":final["convergiu"],"segundos":final["segundos"],"order":melhor.order,
        "seasonal_order":melhor.seasonal_order,"familia":melhor.familia,"exogenas":melhor.exogenas}
display(pd.Series(resumo,name="valor").to_frame())

## 4. Previsão e resíduos individuais SARIMAX

In [ ]:
fig,ax=plt.subplots(figsize=(13,5)); ax.plot(previsoes.index,previsoes.y_true,label="Real",color="black"); ax.plot(previsoes.index,previsoes.y_pred_sarimax,label="SARIMAX",color="#e25c3e"); ax.legend(); ax.set_title(f"Base {BASE_NUMBER} — previsão versus real"); plt.show()
display(previsoes.head(20))

## 5. ACF e diagnóstico dos resíduos SARIMAX

In [ ]:
residuos=previsoes.residuo_sarimax.dropna(); fig,ax=plt.subplots(2,2,figsize=(13,8)); ax[0,0].plot(residuos); ax[0,0].axhline(0,color="black",ls="--"); plot_acf(residuos,lags=min(ACF_LAGS,len(residuos)//3),ax=ax[0,1]); stats.probplot(residuos,dist="norm",plot=ax[1,0]); ax[1,1].hist(residuos,bins=30,density=True); plt.tight_layout(); plt.show()

## 6. Ljung–Box SARIMAX

In [ ]:
lags=[x for x in LJUNG_LAGS if x<len(residuos)]; ljung=acorr_ljungbox(residuos,lags=lags,return_df=True); ljung["rejeita_ruido_branco_5pct"]=ljung.lb_pvalue<.05; display(ljung)

## 7. Coeficientes das variáveis exógenas

In [ ]:
parametros=final["resultado"].params; coef=parametros[parametros.index.isin(EXOG_COLS)].to_frame("coeficiente"); display(coef if len(coef) else pd.DataFrame({"observação":["Modelo selecionado sem exógenas."]}))